In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet("../data/df_taxi_parquet.parquet")

In [3]:
df.columns

Index(['Unnamed: 0', 'Trip ID', 'Taxi ID', 'Trip Start Timestamp',
       'Trip End Timestamp', 'Trip Seconds', 'Trip Miles',
       'Pickup Community Area', 'Dropoff Community Area', 'Fare', 'Tips',
       'Tolls', 'Extras', 'Trip Total', 'Payment Type', 'Company',
       'start_hour', 'start_weekday', 'start_month', 'start_date',
       'is_weekend', 'period_of_day', 'hour_sin', 'hour_cos', 'is_night',
       'Payment Simplified', 'Company_clean', 'company_freq', 'pickup_freq',
       'dropoff_freq', 'speed_mph'],
      dtype='object')

In [5]:
df.columns

Index(['Unnamed: 0', 'Trip ID', 'Taxi ID', 'Trip Start Timestamp',
       'Trip End Timestamp', 'Trip Seconds', 'Trip Miles',
       'Pickup Community Area', 'Dropoff Community Area', 'Fare', 'Tips',
       'Tolls', 'Extras', 'Trip Total', 'Payment Type', 'Company',
       'start_hour', 'start_weekday', 'start_month', 'start_date',
       'is_weekend', 'period_of_day', 'hour_sin', 'hour_cos', 'is_night',
       'Payment Simplified', 'Company_clean', 'company_freq', 'pickup_freq',
       'dropoff_freq', 'speed_mph'],
      dtype='object')

In [9]:
df["Payment Simplified"]

index
0           Other
1           Other
2           Other
3            Card
4            Card
            ...  
12204104     Card
12204107     Card
12204108     Card
12204109     Card
12204110    Other
Name: Payment Simplified, Length: 10814783, dtype: object

In [ ]:
features = [
    'Trip Seconds',
    'Trip Miles',
    'Pickup Community Area',
    'Dropoff Community Area',
    'start_hour',      # OU hour_sin/hour_cos
    'start_weekday',
    'start_month',
    'is_weekend',
    'is_night',
    # 'period_of_day',
    # 'payment_freq',
    'company_freq',
    'pickup_freq',
    'dropoff_freq',
    'speed_mph'
]
# 🔴 Timestamps bruts = dangereux

# Ils contiennent :

# le temps exact dans la vraie vie

# le temps exact dans le dataset (qui n’a rien à voir avec le prix)

# ➡️ Le modèle peut apprendre une tendance artificielle.

features_2 = [
    'Trip Seconds',
    'Trip Miles',
    'Pickup Community Area',
    'Dropoff Community Area',
    'hour_sin',      # OU hour_sin/hour_cos
    'hour_cos',
    'start_weekday',
    'start_month',
    'is_weekend',
    'is_night',
    # 'period_of_day',
    # 'payment_freq',
    'company_freq',
    'pickup_freq',
    'dropoff_freq',
    'speed_mph'
]

target = ["Fare"]
# supp ['Tips', 'Tolls', 'Extras', 'Trip Total']
# innutile : ['Unnamed: 0','Trip ID','Taxi ID','Trip Start Timestamp','Trip End Timestamp','start_date','Payment Type','Payment Simplified','Company','Company_clean']


In [27]:
X = df[features_2]
y = df[target]

In [28]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# === 1. Train / Test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# === 2. Standardisation ===
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# === 3. ElasticNet ===
model = ElasticNet(
    alpha=0.1,
    l1_ratio=0.5,
    max_iter=5000,
    random_state=42
)

# === 4. Metrics sur TRAIN (après fit) ===
model.fit(X_train_scaled, y_train)
y_train_pred = model.predict(X_train_scaled)

train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_r2 = r2_score(y_train, y_train_pred)

print("\n===== TRAIN =====")
print("MAE :", train_mae)
print("RMSE :", train_rmse)
print("R² :", train_r2)

# === 5. Cross-validation sur TRAIN ===
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

cv_mae = -cross_val_score(model, X_train_scaled, y_train, scoring="neg_mean_absolute_error", cv=kfold)
cv_rmse = np.sqrt(-cross_val_score(model, X_train_scaled, y_train, scoring="neg_mean_squared_error", cv=kfold))
cv_r2 = cross_val_score(model, X_train_scaled, y_train, scoring="r2", cv=kfold)

print("\n===== CROSS-VALIDATION (5-fold) =====")
print("MAE :", cv_mae.mean())
print("RMSE :", cv_rmse.mean())
print("R² :", cv_r2.mean())

# === 6. Metrics sur TEST ===
y_test_pred = model.predict(X_test_scaled)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

print("\n===== TEST =====")
print("MAE :", test_mae)
print("RMSE :", test_rmse)
print("R² :", test_r2)



===== TRAIN =====
MAE : 2.0066946495623106
RMSE : 4.884748957010559
R² : 0.9135854176780164

===== CROSS-VALIDATION (5-fold) =====
MAE : 2.0067158143056765
RMSE : 4.884878651274354
R² : 0.9135801295065822

===== TEST =====
MAE : 2.0024518939857727
RMSE : 4.858930263347481
R² : 0.9145309797879471
